In [1]:
pwd

'c:\\Users\\ADMIN\\Documents\\projects\\python'

In [ ]:
# 퀘스트 1 - 은행 계좌
import random
import time

class Bank:

    def __init__(self):
        self.customer_db = {
            "김종하": {
                "계좌 번호": 11111111111,
                "잔고": 6000000,
                "입금 내역": [],
                "출금 내역": []
            }
        }
        # 중복방지를 위한 로직을 짜던 중 딕셔너리에서 직접 찾기보다는 따로 중복방지용 자료구조를 만드는 것이 더 탐색에 유리하다고 판단
        # 찾아본 결과 list보다는 set이 해시테이블형 구조로 탐색에 유리한 조회 구조임을 확인
        self.number_db = {11111111111}

    # Q6 - 고객정보 출력하기
    def display_customer_info(self, name):
        name_form = f"예금주: {name}"
        number_form = f"계좌 번호: {str(self.customer_db[name]["계좌 번호"])[0:3]}-{str(self.customer_db[name]["계좌 번호"])[3:5]}-{str(self.customer_db[name]["계좌 번호"])[5:]}"
        
        # 잔고에 쉼표찍기 병림픽
        # not callable 메시지에 대해 배움
        num_of_comma = len(str(self.customer_db[name]["잔고"])) // 3 # 7이라고 가정시 2
        num_of_remainder = len(str(self.customer_db[name]["잔고"])) % 3 # 7이라고 가정시 1
        temp_list = []
        if num_of_comma > 0:
            for i in range(num_of_comma):
                temp_list.append(str(self.customer_db[name]["잔고"])[::-1][3 * i : 3 * (i + 1)])
            if num_of_remainder != 0:
                temp_list.append(str(self.customer_db[name]["잔고"])[::-1][3 * (num_of_comma) : 3 * (num_of_comma) + num_of_remainder])
            balance_with_comma = ",".join(temp_list)[::-1]
            balance_form = f"잔고: {balance_with_comma}원"
        else:
            balance_form = f"잔고: {str(self.customer_db[name]["잔고"])}원"
        
        print(name_form)
        print(number_form)
        print(balance_form)
    
# Q1 - Account 클래스 만들기
class Account:
    
    # Q2 - Account 클래스 변수 활용하기
    # 계좌번호 생성 횟수 초기화
    account_count = 0
    

    def __init__(self, name, balance):
        # Q3 - Account 클래스로 만든 객체 추적
        Account.account_count += 1
        self.bank = "SC은행"
        self.name = name
        self.number = 0
        self.balance = balance
        # 입금 카운트 초기화
        self.deposit_count = 0
        # Q10 - 입금, 출금 내역
        self.deposit_history = []
        self.withdraw_history = []

    # 고객 가입 메서드(계좌번호 추가 메서드)
    def add_customer(self, bank):
        random_account_number = random.randint(10000000000, 99999999999)
        while True:
            if random_account_number not in bank.number_db:
                self.number = random_account_number
                break
            else:
                random_account_number = random.randint(10000000000, 99999999999)

        # 중복을 피해 계좌번호 부여                    
        # 처음에는 가장 친숙한 update함수를 사용하려고 했으나 인플레이스 함수는 None을 반환하므로 객체에 할당할 수가 없다는 것을 확인
        # list처럼 += 연산자를 쓸 수 없지만 파이썬의 딕셔너리는 새로운 키를 subscripting하는 객체를 만들어 값을 할당하면 자체적으로 그 새로운 키를 추가하고 값을 할당할 수 있음
        bank.customer_db[self.name] = {
            "계좌 번호": self.number,
            "잔고": self.balance
        }
        bank.number_db.add(self.number)

    # Q3 - Account 클래스 변수를 받아와 출력
    def get_account_num(self):
        print(f"계좌 생성 횟수는: {Account.account_count}번 입니다.")

    # Q4 - deposit
    def deposit(self, bank, name, number, liquidity: int):
        if name in bank.customer_db and liquidity > 0 and number == bank.customer_db[name]["계좌 번호"]:
            bank.customer_db[name]["잔고"] += liquidity

            # 입금 시간
            deposit_time = time.strftime('%Y-%m-%d %H:%M:%S')

            # 입금 횟수
            self.deposit_count += 1

            # Q7 - 이자 지급
            if self.deposit_count % 5 == 0:
                bank.customer_db[name]["잔고"] += int(bank.customer_db[name]["잔고"] * 0.01)

            print(f"{liquidity}원 입금완료. {name}님의 계좌 잔액은 {bank.customer_db[name]["잔고"]}원 입니다.")

            # 입금 내역        
            deposit_history = f"[{deposit_time}] [이름: {name}, 입금액: {liquidity}원, 잔액:{bank.customer_db[name]["잔고"]}]"
            bank.customer_db[name]["입금 내역"] += [deposit_history]
        else:
            print("잘못된 정보 입력으로 계좌 접근이 거부되었습니다.")

        

    # Q5 - withdraw
    def withdraw(self, bank, name, number, liquidity: int):
        if name in bank.customer_db and liquidity > 0 and liquidity <= bank.customer_db[name]["잔고"] and number == bank.customer_db[name]["계좌 번호"]:
            bank.customer_db[name]["잔고"] -= liquidity
            print(f"{liquidity}원 출금완료. {name}님의 계좌 잔액은 {bank.customer_db[name]["잔고"]}원 입니다.")
            withdraw_time = time.strftime('%Y-%m-%d %H:%M:%S')

            # 출금 내역
            withdraw_history = f"[{withdraw_time}] [이름: {name}, 출금액: {liquidity}원, 잔액:{bank.customer_db[name]["잔고"]}]"
            bank.customer_db[name]["출금 내역"] += [withdraw_history]
        else:
            print("잘못된 정보 입력으로 계좌 접근이 거부되었습니다.") # 정밀하게 수정 필요


In [103]:
def create_acoount(name, balance):
    create_account = Account(name, balance)
    create_account.add_customer(bank) 
    bank.display_customer_info(name)
    # print(bank.customer_db)
    create_account.get_account_num()

# 드디어 해냈다! 
bank = Bank()
while True:
    name = input("이름을 입력해주세요: ")
    if name == "exit":
        break
    string_balance = input("잔고를 입력해주세요: ")
    if string_balance == "exit":
        break
    balance = int(string_balance)
    run(name=name, balance=balance)

예금주: 김성훈
계좌 번호: 270-95-086032
잔고: 100,000원
계좌 생성 횟수는: 1번 입니다.


In [ ]:
# Q1. Account 클래스 : 은행에 가서 계좌를 개설하면 은행이름, 예금주, 계좌번호, 잔액이 설정됩니다. 
#     Account 클래스를 생성한 후 생성자(hint: 매직메서드...!!)를 구현해보세요. 
#     생성자에서는 예금주와 초기 잔액만 입력 받습니다. 
#     은행이름은 SC은행으로 계좌번호는 3자리-2자리-6자리 형태로 랜덤하게 생성됩니다. 
#     (은행이름: SC은행, 계좌번호: 111-11-111111)

# Q2. 클래스 변수 : 클래스 변수를 사용해서 Account 클래스로부터 생성된 계좌 객체의 개수를 저장하세요.

# Q3. 클래스 변수 출력 : Account 클래스로부터 생성된 계좌의 개수를 출력하는 get_account_num() 메서드를 추가하세요.

# Q4. 입금 메서드 : Account 클래스에 입금을 위한 deposit 메서드를 추가하세요. 입금은 최소 1원 이상만 가능합니다.

# Q5. 출금 메서드 : Account 클래스에 출금을 위한 withdraw 메서드를 추가하세요. 
#     출금은 계좌의 잔고 이상으로 출금할 수는 없습니다.

# Q6. 정보 출력 메서드 : Account 인스턴스에 저장된 정보를 출력하는 display_info() 메서드를 추가하세요. 
#     잔고는 세자리마다 쉼표를 출력하세요.
#     (은행이름: SC은행, 예금주: 파이썬, 계좌번호: 111-11-111111, 잔고: 10,000원)

# Q7. 이자 지급하기 : 입금 횟수가 5회가 될 때 잔고를 기준으로 1%의 이자가 잔고에 추가되도록 코드를 변경해보세요.

# Q8. 여러 객체 생성 : Account 클래스로부터 3개 이상 인스턴스를 생성하고 생성된 인스턴스를 리스트에 저장해보세요.

# Q9. 객체 순회 : 반복문을 통해 리스트에 있는 객체를 순회하면서 잔고가 100만원 이상인 고객의 정보만 출력하세요.

# Q10. 입출금 내역 기록 : 입금과 출금 내역이 기록되도록 코드를 업데이트 하세요.
#      (입금 내역과 출금 내역을 출력하는 deposit_history와 withdraw_history 메서드를 추가하세요.)